# 01 — Getting Started with PySpark

You only have one laptop, and that's fine. Spark's "parallel computing" story is really about splitting data into **partitions** and processing them with multiple **worker threads/executors**. On a single machine you run in **local mode**, where Spark simulates a cluster using the CPU cores you already have (e.g. `local[*]` = use all available cores). The exact same DataFrame code you write here would run unchanged on a real multi-node cluster later — that's the whole point of learning the API this way.

This notebook covers:
1. Creating a `SparkSession`
2. Loading a CSV into a DataFrame
3. Inspecting schema and data
4. Basic transformations (`select`, `filter`, `withColumn`)
5. Aggregations (`groupBy`)
6. Running SQL directly on a DataFrame
7. Stopping the session

## 1. Create a SparkSession

The `SparkSession` is your entry point to everything in Spark. `local[*]` tells Spark to run locally using all available CPU cores as "workers" — this is what gives you parallelism even without a cluster.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("getting-started")
    .master("local[*]")
    .getOrCreate()
)

spark

Click the Spark UI link above (or open http://localhost:4040 on your Mac) while this session is active — it shows jobs, stages, and tasks so you can literally watch the parallel execution happen.

## 2. Load data into a DataFrame

We'll use the sample `data/orders.csv` file mounted into the container. The notebook's working directory is `notebooks/`, so we reference it as `../data/orders.csv`. A Spark DataFrame looks like a pandas DataFrame but is lazily evaluated and can be split across partitions.

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/orders.csv")
)

df.printSchema()
df.show(5)

Useful checks: how many partitions is this DataFrame split into, and how many rows total?

In [ ]:
print("Partitions:", df.rdd.getNumPartitions())
print("Row count:", df.count())

## 3. Basic transformations

Transformations (`select`, `filter`, `withColumn`, ...) are **lazy** — Spark just builds a plan. Nothing actually runs until you call an **action** like `.show()`, `.count()`, or `.collect()`.

In [ ]:
# select a subset of columns
df.select("order_id", "customer", "region", "category").show(5)

In [ ]:
# filter rows
df.filter(df.region == "SG").show(5)

In [ ]:
from pyspark.sql import functions as F

# add a derived column
df_with_total = df.withColumn("total_price", F.round(df.quantity * df.unit_price, 2))
df_with_total.select("order_id", "product", "quantity", "unit_price", "total_price").show(5)

## 4. Aggregations

`groupBy` + `agg` is the bread and butter of Spark analytics — same mental model as SQL `GROUP BY` or pandas `groupby`.

In [ ]:
revenue_by_region = (
    df_with_total.groupBy("region")
    .agg(
        F.sum("total_price").alias("total_revenue"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.desc("total_revenue"))
)

revenue_by_region.show()

In [ ]:
revenue_by_category = (
    df_with_total.groupBy("category")
    .agg(F.sum("total_price").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

revenue_by_category.show()

## 5. SQL, if you prefer it

You can register a DataFrame as a temporary view and query it with plain SQL — same engine, same execution plan, just a different syntax on top.

In [ ]:
df_with_total.createOrReplaceTempView("orders")

spark.sql(
    """
    SELECT region, ROUND(SUM(total_price), 2) AS total_revenue
    FROM orders
    GROUP BY region
    ORDER BY total_revenue DESC
    """
).show()

## 6. Stop the session

Always stop your SparkSession when you're done with a notebook — it frees the JVM resources it's holding.

In [ ]:
spark.stop()

## Try it yourself

Before moving to the next notebook, re-run the SparkSession cell and try:
1. Find the single most popular `product` by total `quantity` sold.
2. Find the average order value (`total_price`) per `region`.
3. Filter to orders in February 2026 only, and show them sorted by `total_price` descending.
4. Repartition the DataFrame with `df.repartition(4)` and check `getNumPartitions()` again — what changed?